# FNO Model 3 (Omega + 3 Channels, 4 Total)

Low-resolution train/validation + high-resolution test (zero-shot super-resolution) notebook.


In [ ]:
# =========================
# Imports and experiment configuration
# =========================
# Standard library helpers for filesystem operations, frame sorting, and reproducibility.
import glob
import random
from pathlib import Path
from collections import defaultdict

# Numerical + plotting + torch utilities.
import numpy as np
import torch
import matplotlib.pyplot as plt
from torch.utils.data import Dataset, DataLoader

# Neural operator primitives for FNO training.
from neuralop.models import FNO
from neuralop.losses import LpLoss
from neuralop.utils import count_model_params

# Main configuration dictionary.
# You only need to edit values here to switch between random/chronological split,
# train/test directories, and low-res/high-res settings.
CFG = {
    'seed': 42,

    # ---------- Data directories ----------
    # Use low-resolution data directories here (for train+val split only).
    'trainval_dirs': [
        'data/train/pair_1_grid_2D',
        'data/train/pair_2_grid_2D',
    ],
    # Use separate high-resolution test directories here (zero-shot SR test set).
    'test_dirs': [
        'data/test/pair_1_grid_2D',
        'data/test/pair_2_grid_2D',
    ],

    # ---------- Split behavior ----------
    # Set to 'random' (default) or 'chronological'.
    'split_mode': 'random',
    'train_frac': 0.8,

    # ---------- Resolution expectations ----------
    # Set these to your target setup for zero-shot super-resolution.
    'train_expected_res': 16,
    'test_expected_res': 64,

    # ---------- Runtime behavior ----------
    # If True, notebook continues even when test directories are missing.
    # If False, it raises an error when test data is unavailable.
    'allow_missing_test_dirs': True,

    'batch_size': 4,
    'epochs': 120,
    'lr': 1e-3,
    'weight_decay': 1e-4,

    # For 16x16 training, keep modes <= 8 (Nyquist safety). 6 is a safe default.
    'n_modes': (6, 6),
    'hidden_channels': 64,
    'projection_channel_ratio': 4,

    'eval_every': 1,

    # Variant-specific input definition.
    'input_channels': 4,
    'input_key': 'input_grid_omega4',
    'file_tag': 'fno_model3',
}

# Reproducibility helper.
def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

set_seed(CFG['seed'])
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)
print('Variant:', CFG['input_key'], 'channels=', CFG['input_channels'])
print('Split mode:', CFG['split_mode'])


In [ ]:
# =========================
# Data setup (train/val low-res + test high-res)
# =========================
# Resolve repo root safely from notebook execution directory.
cwd = Path.cwd().resolve()
if (cwd / 'data').exists():
    REPO_ROOT = cwd
elif (cwd.parent / 'data').exists():
    REPO_ROOT = cwd.parent
else:
    raise FileNotFoundError('Could not locate repo root containing data/')

RESULTS_DIR = REPO_ROOT / 'results'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('Results directory:', RESULTS_DIR)

# Figure save helper.
def save_fig(fig, filename, dpi=220):
    out = RESULTS_DIR / f"{CFG['file_tag']}_{filename}"
    fig.savefig(out, dpi=dpi, bbox_inches='tight')
    print(f'[saved] {out}')

# Turn relative folder strings into absolute directories.
def resolve_dirs(rel_dirs, must_exist=True):
    dirs = []
    for d in rel_dirs:
        p = (REPO_ROOT / d).resolve()
        if p.is_dir():
            dirs.append(p)
        elif must_exist:
            raise FileNotFoundError(f'Missing directory: {p}')
        else:
            print(f'[warn] test directory missing, skipping: {p}')
    return dirs

TRAINVAL_DIRS = resolve_dirs(CFG['trainval_dirs'], must_exist=True)
TEST_DIRS = resolve_dirs(CFG['test_dirs'], must_exist=not CFG['allow_missing_test_dirs'])

# Read frame files in sorted order by frame index.
def list_frame_files(sim_dir):
    files = sorted(
        glob.glob(str(sim_dir / 'frame_*_grid.npz')),
        key=lambda f: int(Path(f).stem.split('_')[1]),
    )
    return files

# Split one simulation into train/val based on selected mode.
def split_train_val(files, train_frac=0.8, mode='random', seed=42):
    files = list(files)
    n = len(files)
    if n <= 1:
        raise RuntimeError('Need at least 2 frames for train/val split')

    if mode == 'random':
        rng = np.random.default_rng(seed)
        idx = np.arange(n)
        rng.shuffle(idx)
        files = [files[i] for i in idx]
    elif mode == 'chronological':
        pass
    else:
        raise ValueError("split_mode must be 'random' or 'chronological'")

    n_train = max(1, int(train_frac * n))
    n_train = min(n_train, n - 1)
    return files[:n_train], files[n_train:]

# Build global train/val lists from low-res directories.
TRAIN_FILES, VAL_FILES = [], []
for i, d in enumerate(TRAINVAL_DIRS):
    files = list_frame_files(d)
    tr, va = split_train_val(
        files,
        train_frac=CFG['train_frac'],
        mode=CFG['split_mode'],
        seed=CFG['seed'] + i,
    )
    TRAIN_FILES.extend(tr)
    VAL_FILES.extend(va)

# Build test list from separate high-res directories.
TEST_FILES = []
for d in TEST_DIRS:
    TEST_FILES.extend(list_frame_files(d))

HAS_TEST = len(TEST_FILES) > 0
if not HAS_TEST:
    print('[warn] No test files found. Test metrics/plots will be skipped.')

print(f'Train files: {len(TRAIN_FILES)} | Val files: {len(VAL_FILES)} | Test files: {len(TEST_FILES)}')

# Input loader for selected channel key.
def load_x(data):
    key = CFG['input_key']
    if key not in data:
        raise KeyError(f"Missing key '{key}'. Available: {data.files}")
    x = np.asarray(data[key], dtype=np.float32)
    if x.ndim == 2:
        x = x[None, :, :]
    return x

# Shape/resolution assertions.
def assert_shapes(files, expected_channels, expected_res=None, label='set'):
    if len(files) == 0:
        return
    bad = []
    for f in files:
        d = np.load(f)
        x = load_x(d)
        y = np.asarray(d['U_grid'], dtype=np.float32)

        ok_channels = (x.shape[0] == expected_channels and y.shape[0] == 2)
        ok_res = True
        if expected_res is not None:
            ok_res = (
                x.shape[-2:] == (expected_res, expected_res)
                and y.shape[-2:] == (expected_res, expected_res)
            )

        if not (ok_channels and ok_res):
            bad.append((f, tuple(x.shape), tuple(y.shape)))
            if len(bad) >= 5:
                break

    if bad:
        msg = '\n'.join([f"{fp} -> x:{xs}, y:{ys}" for fp, xs, ys in bad])
        raise RuntimeError(f'{label} shape check failed. Examples:\n{msg}')

# Leakage guard.
def assert_no_overlap(a, b):
    inter = set(a).intersection(set(b))
    if inter:
        raise RuntimeError(f'Train/val leakage detected with {len(inter)} overlapping files')

assert_no_overlap(TRAIN_FILES, VAL_FILES)
assert_shapes(TRAIN_FILES, expected_channels=CFG['input_channels'], expected_res=CFG['train_expected_res'], label='train')
assert_shapes(VAL_FILES, expected_channels=CFG['input_channels'], expected_res=CFG['train_expected_res'], label='val')
if HAS_TEST:
    assert_shapes(TEST_FILES, expected_channels=CFG['input_channels'], expected_res=CFG['test_expected_res'], label='test')

# Dataset object used for all three splits.
class GridDataset(Dataset):
    def __init__(self, files):
        self.files = files

    def __len__(self):
        return len(self.files)

    def __getitem__(self, idx):
        d = np.load(self.files[idx])
        x = torch.tensor(load_x(d), dtype=torch.float32)
        y = torch.tensor(np.asarray(d['U_grid'], dtype=np.float32), dtype=torch.float32)
        return x, y

TRAIN_DS = GridDataset(TRAIN_FILES)
VAL_DS = GridDataset(VAL_FILES)
TEST_DS = GridDataset(TEST_FILES) if HAS_TEST else None

# Data loaders.
drop_last = len(TRAIN_DS) >= CFG['batch_size']
TRAIN_LOADER = DataLoader(TRAIN_DS, batch_size=CFG['batch_size'], shuffle=True, drop_last=drop_last)
VAL_LOADER = DataLoader(VAL_DS, batch_size=CFG['batch_size'], shuffle=False)
TEST_LOADER = DataLoader(TEST_DS, batch_size=CFG['batch_size'], shuffle=False) if HAS_TEST else None

# Print one sample from each split to make resolution differences explicit.
xtr, ytr = TRAIN_DS[0]
xva, yva = VAL_DS[0]
print('Train sample shape:', tuple(xtr.shape), tuple(ytr.shape))
print('Val sample shape  :', tuple(xva.shape), tuple(yva.shape))
if HAS_TEST:
    xte, yte = TEST_DS[0]
    print('Test sample shape :', tuple(xte.shape), tuple(yte.shape))


In [ ]:
# =========================
# Model + optimizer + metrics
# =========================
def build_model(cfg):
    return FNO(
        n_modes=cfg['n_modes'],
        in_channels=cfg['input_channels'],
        out_channels=2,
        hidden_channels=cfg['hidden_channels'],
        projection_channel_ratio=cfg['projection_channel_ratio'],
    ).to(DEVICE)

MODEL = build_model(CFG)

total_params = count_model_params(MODEL)
trainable_params = sum(p.numel() for p in MODEL.parameters() if p.requires_grad)
print('Total params:', total_params)
print('Trainable params:', trainable_params)

OPT = torch.optim.AdamW(MODEL.parameters(), lr=CFG['lr'], weight_decay=CFG['weight_decay'])
SCH = torch.optim.lr_scheduler.CosineAnnealingLR(OPT, T_max=CFG['epochs'])
LOSS_FN = LpLoss(d=2, p=2)

def predict(model, x):
    return model(x)

def batch_metrics(pred, target):
    diff = pred - target
    mse = torch.mean(diff ** 2, dim=(1,2,3))
    mae = torch.mean(torch.abs(diff), dim=(1,2,3))
    rmse = torch.sqrt(mse)

    diff_flat = diff.reshape(diff.shape[0], -1)
    tgt_flat = target.reshape(target.shape[0], -1)
    rel_l2 = torch.linalg.norm(diff_flat, dim=1) / torch.linalg.norm(tgt_flat, dim=1).clamp_min(1e-12)

    return {
        'mse': mse.mean().item(),
        'mae': mae.mean().item(),
        'rmse': rmse.mean().item(),
        'rel_l2': rel_l2.mean().item(),
        'rel_vec': rel_l2.detach().cpu().numpy(),
    }

@torch.no_grad()
def evaluate(model, loader, collect=False):
    if loader is None:
        return None
    model.eval()
    acc = {'mse':0.0,'mae':0.0,'rmse':0.0,'rel_l2':0.0}
    rels = []
    nb = 0
    for x,y in loader:
        x,y = x.to(DEVICE), y.to(DEVICE)
        pred = predict(model, x)
        m = batch_metrics(pred, y)
        for k in acc:
            acc[k] += m[k]
        if collect:
            rels.extend(m['rel_vec'].tolist())
        nb += 1
    for k in acc:
        acc[k] /= max(nb,1)
    if collect:
        acc['rel_per_sample'] = np.asarray(rels, dtype=np.float32)
    return acc


In [ ]:
# =========================
# Training loop (train@low-res, val@low-res, test@high-res)
# =========================
HIST = {
    'epoch': [],
    'train_lp': [],
    'val_rel_l2': [],
    'val_rmse': [],
    'test_rel_l2': [],
    'test_rmse': [],
    'lr': [],
}

best_val = float('inf')
CKPT_PATH = RESULTS_DIR / f"{CFG['file_tag']}_best_model.pt"

for epoch in range(1, CFG['epochs'] + 1):
    MODEL.train()
    run_loss = 0.0
    nb = 0

    for x,y in TRAIN_LOADER:
        x,y = x.to(DEVICE), y.to(DEVICE)
        OPT.zero_grad(set_to_none=True)
        pred = predict(MODEL, x)
        loss = LOSS_FN(pred, y)
        loss.backward()
        OPT.step()
        run_loss += loss.item()
        nb += 1

    SCH.step()
    train_lp = run_loss / max(nb,1)

    if epoch % CFG['eval_every'] == 0:
        val_m = evaluate(MODEL, VAL_LOADER)
        test_m = evaluate(MODEL, TEST_LOADER) if HAS_TEST else None

        HIST['epoch'].append(epoch)
        HIST['train_lp'].append(train_lp)
        HIST['val_rel_l2'].append(val_m['rel_l2'])
        HIST['val_rmse'].append(val_m['rmse'])
        HIST['test_rel_l2'].append(test_m['rel_l2'] if test_m is not None else np.nan)
        HIST['test_rmse'].append(test_m['rmse'] if test_m is not None else np.nan)
        HIST['lr'].append(OPT.param_groups[0]['lr'])

        if val_m['rel_l2'] < best_val:
            best_val = val_m['rel_l2']
            torch.save({
                'model_state_dict': MODEL.state_dict(),
                'config': dict(CFG),
                'best_val_rel_l2': best_val,
            }, CKPT_PATH)

        msg = (
            f"[{epoch:03d}] train_lp={train_lp:.4e} "
            f"val_rel_l2={val_m['rel_l2']:.4e} "
            f"val_rmse={val_m['rmse']:.4e} "
            f"lr={HIST['lr'][-1]:.2e}"
        )
        if test_m is not None:
            msg += f" test_rel_l2={test_m['rel_l2']:.4e} test_rmse={test_m['rmse']:.4e}"
        print(msg)

print(f'Best val rel-L2: {best_val:.4e}')
print(f'Checkpoint: {CKPT_PATH}')


In [ ]:
# =========================
# Post-training test snippet (load best checkpoint and test)
# =========================
# This cell is intentionally separate from the training loop.
# It verifies test performance using the saved best-validation checkpoint.
if not HAS_TEST:
    print("[skip] No test split available. Fill CFG['test_dirs'] and rerun.")
else:
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    MODEL.load_state_dict(ckpt['model_state_dict'])
    MODEL.eval()

    test_m = evaluate(MODEL, TEST_LOADER, collect=True)
    print('Loaded checkpoint:', CKPT_PATH)
    print('Checkpoint best val rel-L2:', ckpt.get('best_val_rel_l2', 'n/a'))
    print(
        f"Post-train TEST metrics -> rel-L2={test_m['rel_l2']:.4e}, "
        f"RMSE={test_m['rmse']:.4e}, MAE={test_m['mae']:.4e}, MSE={test_m['mse']:.4e}"
    )

    # One qualitative test example.
    tx, ty = TEST_DS[0]
    with torch.no_grad():
        tp = predict(MODEL, tx.unsqueeze(0).to(DEVICE))[0].cpu()

    t_true = torch.sqrt(ty[0] ** 2 + ty[1] ** 2)
    t_pred = torch.sqrt(tp[0] ** 2 + tp[1] ** 2)
    t_err = torch.abs(t_pred - t_true)

    fig_test = plt.figure(figsize=(12, 3.8))
    plt.subplot(1, 3, 1); plt.imshow(t_true, cmap='turbo'); plt.title('TEST true speed'); plt.colorbar()
    plt.subplot(1, 3, 2); plt.imshow(t_pred, cmap='turbo'); plt.title('TEST pred speed'); plt.colorbar()
    plt.subplot(1, 3, 3); plt.imshow(t_err, cmap='magma'); plt.title('TEST |error|'); plt.colorbar()
    fig_test.tight_layout()
    save_fig(fig_test, 'posttrain_test_sample.png')
    plt.show()


In [ ]:
# =========================
# Main plots: convergence + zero-shot super-resolution evidence
# =========================
# Parameter distribution plot.
module_param_counts = defaultdict(int)
for name, p in MODEL.named_parameters():
    module_param_counts[name.split('.')[0]] += p.numel()

mods = list(module_param_counts.keys())
vals = [module_param_counts[m] for m in mods]
order = np.argsort(vals)[::-1]
mods = [mods[i] for i in order]
vals = [vals[i] for i in order]

fig_p, ax_p = plt.subplots(figsize=(10,4))
ax_p.bar(mods, vals)
ax_p.set_title(f"Parameter distribution (total={total_params:,})")
ax_p.set_ylabel('Parameters')
ax_p.tick_params(axis='x', rotation=45)
fig_p.tight_layout()
save_fig(fig_p, 'params_by_module.png')
plt.show()

# Convergence with explicit low-res vs high-res curves.
fig_c, axs = plt.subplots(1,3, figsize=(16,4))
axs[0].plot(HIST['epoch'], HIST['train_lp'], label='train Lp (16x16)')
axs[0].plot(HIST['epoch'], HIST['val_rel_l2'], label='val rel-L2 (16x16)')
if HAS_TEST:
    axs[0].plot(HIST['epoch'], HIST['test_rel_l2'], label='test rel-L2 (64x64)')
axs[0].set_yscale('log')
axs[0].set_xlabel('Epoch')
axs[0].set_title('Convergence (Zero-shot SR setup)')
axs[0].legend()

axs[1].plot(HIST['epoch'], HIST['val_rmse'], label='val RMSE (16x16)')
if HAS_TEST:
    axs[1].plot(HIST['epoch'], HIST['test_rmse'], label='test RMSE (64x64)')
axs[1].set_yscale('log')
axs[1].set_xlabel('Epoch')
axs[1].set_title('RMSE by split/resolution')
axs[1].legend()

axs[2].plot(HIST['epoch'], HIST['lr'], label='LR')
axs[2].set_xlabel('Epoch')
axs[2].set_title('Learning-rate schedule')
axs[2].legend()

fig_c.tight_layout()
save_fig(fig_c, 'convergence_zero_shot_sr.png')
plt.show()

# Qualitative comparison: validation low-res vs test high-res.
MODEL.eval()
fig_q = plt.figure(figsize=(15, 6 if HAS_TEST else 3.5))

# Low-resolution validation sample.
vx, vy = VAL_DS[0]
with torch.no_grad():
    vp = predict(MODEL, vx.unsqueeze(0).to(DEVICE))[0].cpu()

v_true = torch.sqrt(vy[0]**2 + vy[1]**2)
v_pred = torch.sqrt(vp[0]**2 + vp[1]**2)
v_err = torch.abs(v_pred - v_true)

plt.subplot(2 if HAS_TEST else 1, 3, 1); plt.imshow(v_true, cmap='turbo'); plt.title('VAL true speed (low-res)'); plt.colorbar()
plt.subplot(2 if HAS_TEST else 1, 3, 2); plt.imshow(v_pred, cmap='turbo'); plt.title('VAL pred speed (low-res)'); plt.colorbar()
plt.subplot(2 if HAS_TEST else 1, 3, 3); plt.imshow(v_err, cmap='magma'); plt.title('VAL |error| (low-res)'); plt.colorbar()

if HAS_TEST:
    tx, ty = TEST_DS[0]
    with torch.no_grad():
        tp = predict(MODEL, tx.unsqueeze(0).to(DEVICE))[0].cpu()
    t_true = torch.sqrt(ty[0]**2 + ty[1]**2)
    t_pred = torch.sqrt(tp[0]**2 + tp[1]**2)
    t_err = torch.abs(t_pred - t_true)

    plt.subplot(2, 3, 4); plt.imshow(t_true, cmap='turbo'); plt.title('TEST true speed (high-res)'); plt.colorbar()
    plt.subplot(2, 3, 5); plt.imshow(t_pred, cmap='turbo'); plt.title('TEST pred speed (high-res)'); plt.colorbar()
    plt.subplot(2, 3, 6); plt.imshow(t_err, cmap='magma'); plt.title('TEST |error| (high-res)'); plt.colorbar()

fig_q.tight_layout()
save_fig(fig_q, 'qualitative_low_vs_high_resolution.png')
plt.show()


In [ ]:
# =========================
# Relative error curve + distribution/parity diagnostics
# =========================
if len(HIST['epoch']) == 0:
    raise RuntimeError('History is empty; run training first.')

fig_r, ax_r = plt.subplots(figsize=(7,4))
ax_r.plot(HIST['epoch'], HIST['val_rel_l2'], marker='o', label='val rel-L2 (16x16)')
if HAS_TEST:
    ax_r.plot(HIST['epoch'], HIST['test_rel_l2'], marker='o', label='test rel-L2 (64x64)')
ax_r.set_yscale('log')
ax_r.set_xlabel('Epoch')
ax_r.set_ylabel('Relative L2 error')
ax_r.set_title('Relative error vs epoch')
ax_r.grid(True, alpha=0.3)
ax_r.legend()
fig_r.tight_layout()
save_fig(fig_r, 'relative_error_vs_epoch.png')
plt.show()

# Distribution + parity based on test split when available, else val split.
if HAS_TEST:
    diag_loader = TEST_LOADER
    diag_ds = TEST_DS
    diag_label = 'TEST (64x64)'
else:
    diag_loader = VAL_LOADER
    diag_ds = VAL_DS
    diag_label = 'VAL (16x16)'

diag = evaluate(MODEL, diag_loader, collect=True)
rel_vals = np.asarray(diag['rel_per_sample'], dtype=np.float32)

sx, sy = diag_ds[0]
with torch.no_grad():
    sp = predict(MODEL, sx.unsqueeze(0).to(DEVICE))[0].cpu()

true_speed = torch.sqrt(sy[0]**2 + sy[1]**2).reshape(-1).numpy()
pred_speed = torch.sqrt(sp[0]**2 + sp[1]**2).reshape(-1).numpy()

rng = np.random.default_rng(CFG['seed'])
if true_speed.size > 5000:
    idx = rng.choice(true_speed.size, size=5000, replace=False)
    true_speed = true_speed[idx]
    pred_speed = pred_speed[idx]

fig_d, axs = plt.subplots(1,3, figsize=(16,4))
axs[0].hist(rel_vals, bins=30, color='tab:blue', alpha=0.8)
axs[0].set_title(f'{diag_label} rel-L2 histogram')
axs[0].set_xlabel('rel-L2')
axs[0].set_ylabel('count')

sorted_rel = np.sort(rel_vals)
cdf = np.linspace(0,1,len(sorted_rel),endpoint=True)
axs[1].plot(sorted_rel, cdf, color='tab:green')
axs[1].set_title(f'{diag_label} rel-L2 CDF')
axs[1].set_xlabel('rel-L2')
axs[1].set_ylabel('cdf')
axs[1].grid(True, alpha=0.3)

vmin = min(true_speed.min(), pred_speed.min())
vmax = max(true_speed.max(), pred_speed.max())
axs[2].scatter(true_speed, pred_speed, s=6, alpha=0.35)
axs[2].plot([vmin, vmax], [vmin, vmax], 'r--', linewidth=1.4)
axs[2].set_title(f'Parity plot ({diag_label})')
axs[2].set_xlabel('true speed')
axs[2].set_ylabel('pred speed')

fig_d.tight_layout()
save_fig(fig_d, 'error_distribution_and_parity.png')
plt.show()


In [ ]:
# =========================
# Chronological snapshot panel utility (works for val or test files)
# =========================
def frame_id(path_like):
    return int(Path(path_like).stem.split('_')[1])

def load_xy(file_path):
    d = np.load(file_path)
    x = torch.tensor(load_x(d), dtype=torch.float32)
    y = torch.tensor(np.asarray(d['U_grid'], dtype=np.float32), dtype=torch.float32)
    return x, y

def plot_snapshot_panel(files, panel_name, n_times=4):
    if len(files) == 0:
        print(f'[skip] no files for {panel_name}')
        return

    files = sorted(files, key=frame_id)
    n_times = max(1, min(n_times, len(files)))
    idxs = np.linspace(0, len(files)-1, n_times, dtype=int)
    chosen = [files[i] for i in idxs]

    init_x, _ = load_xy(files[0])

    true_fields, pred_fields, err_fields, rels, tids = [], [], [], [], []
    MODEL.eval()
    with torch.no_grad():
        for f in chosen:
            x,y = load_xy(f)
            p = predict(MODEL, x.unsqueeze(0).to(DEVICE))[0].cpu()

            st = torch.sqrt(y[0]**2 + y[1]**2)
            sp = torch.sqrt(p[0]**2 + p[1]**2)
            se = torch.abs(sp - st)
            rel = (torch.linalg.norm((p-y).reshape(-1)) / torch.linalg.norm(y.reshape(-1)).clamp_min(1e-12)).item()

            true_fields.append(st); pred_fields.append(sp); err_fields.append(se)
            rels.append(rel); tids.append(frame_id(f))

    ncols = 1 + len(chosen)
    fig, axs = plt.subplots(3, ncols, figsize=(3.2*ncols, 9))

    axs[0,0].imshow(init_x[0], cmap='turbo'); axs[0,0].set_title('Initial Input'); axs[0,0].axis('off')
    axs[1,0].axis('off'); axs[1,0].text(0.5,0.5,'Prediction',ha='center',va='center',fontsize=13)
    axs[2,0].axis('off'); axs[2,0].text(0.5,0.5,'Pointwise\n|error|',ha='center',va='center',fontsize=13)

    for j,(fid,tf,pf,ef,rel) in enumerate(zip(tids,true_fields,pred_fields,err_fields,rels), start=1):
        im0=axs[0,j].imshow(tf,cmap='turbo'); axs[0,j].set_title(f't={fid}'); axs[0,j].axis('off'); plt.colorbar(im0,ax=axs[0,j],fraction=0.046,pad=0.04)
        im1=axs[1,j].imshow(pf,cmap='turbo'); axs[1,j].axis('off'); plt.colorbar(im1,ax=axs[1,j],fraction=0.046,pad=0.04)
        im2=axs[2,j].imshow(ef,cmap='magma'); axs[2,j].set_title(f'rel-L2={rel:.3e}'); axs[2,j].axis('off'); plt.colorbar(im2,ax=axs[2,j],fraction=0.046,pad=0.04)

    fig.suptitle(f'{panel_name} snapshots', fontsize=14, y=0.995)
    fig.tight_layout()
    safe = panel_name.replace('/','_').replace(' ','_')
    save_fig(fig, f'snapshots_{safe}.png')
    plt.show()

# Show low-res validation snapshots.
plot_snapshot_panel(VAL_FILES, panel_name='VAL_LOW_RES', n_times=4)

# Show high-res test snapshots (if available).
if HAS_TEST:
    plot_snapshot_panel(TEST_FILES, panel_name='TEST_HIGH_RES', n_times=4)


In [ ]:
# =========================
# Data-hunger curve (train subset size vs validation error)
# =========================
def train_eval_subset(train_files_subset, cfg, quick_epochs=8):
    ds = GridDataset(train_files_subset)
    drop_last = len(ds) >= cfg['batch_size']
    loader = DataLoader(ds, batch_size=cfg['batch_size'], shuffle=True, drop_last=drop_last)

    m = build_model(cfg)
    opt = torch.optim.AdamW(m.parameters(), lr=cfg['lr'], weight_decay=cfg['weight_decay'])
    sch = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=quick_epochs)
    loss_fn = LpLoss(d=2, p=2)

    for _ in range(quick_epochs):
        m.train()
        for x,y in loader:
            x,y = x.to(DEVICE), y.to(DEVICE)
            opt.zero_grad(set_to_none=True)
            p = predict(m, x)
            loss = loss_fn(p,y)
            loss.backward()
            opt.step()
        sch.step()

    vm = evaluate(m, VAL_LOADER)
    return vm['rel_l2']

def run_data_hunger_curve(fractions=(0.1,0.2,0.4,0.6,0.8,1.0), repeats=2, quick_epochs=8):
    rng = np.random.default_rng(CFG['seed'])
    n_total = len(TRAIN_FILES)
    curve = []

    for frac in fractions:
        n_use = max(1, int(frac*n_total))
        vals = []
        for _ in range(repeats):
            idx = rng.choice(n_total, size=n_use, replace=False)
            subset = [TRAIN_FILES[i] for i in sorted(idx.tolist())]
            rel = train_eval_subset(subset, dict(CFG), quick_epochs=quick_epochs)
            vals.append(rel)
        curve.append((frac, n_use, float(np.mean(vals)), float(np.std(vals))))
        print(f'frac={frac:.2f}, n={n_use}, val rel-L2 mean={np.mean(vals):.4e}, std={np.std(vals):.2e}')

    ns = [c[1] for c in curve]
    means = [c[2] for c in curve]
    stds = [c[3] for c in curve]

    fig, ax = plt.subplots(figsize=(7,4))
    ax.errorbar(ns, means, yerr=stds, marker='o', capsize=4)
    ax.set_xlabel('Number of training samples')
    ax.set_ylabel('Validation rel-L2 (low-res val)')
    ax.set_title('Data-hunger curve')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_fig(fig, 'data_hunger_curve.png')
    plt.show()

    return curve

DATA_HUNGER = run_data_hunger_curve()


In [ ]:
# =========================
# Convergence sensitivity sweeps
# =========================
def cfg_with(base_cfg, key, value):
    c = dict(base_cfg)
    c[key] = value
    return c

def quick_sweep(param_name, values, quick_epochs=8):
    results = []
    for v in values:
        c = cfg_with(CFG, param_name, v)
        rel = train_eval_subset(TRAIN_FILES, c, quick_epochs=quick_epochs)
        results.append((v, rel))
        print(f'{param_name}={v} -> val rel-L2 {rel:.4e}')

    xs = [r[0] for r in results]
    ys = [r[1] for r in results]
    fig, ax = plt.subplots(figsize=(6,4))
    ax.plot(xs, ys, marker='o')
    ax.set_xlabel(param_name)
    ax.set_ylabel('Final val rel-L2')
    ax.set_title(f'Quick sweep: {param_name}')
    ax.grid(True, alpha=0.3)
    fig.tight_layout()
    save_fig(fig, f'sweep_{param_name}.png')
    plt.show()

    return results

SWEEP_HIDDEN = quick_sweep('hidden_channels', [32, 48, 64, 96], quick_epochs=8)
SWEEP_LR = quick_sweep('lr', [5e-4, 1e-3, 2e-3], quick_epochs=8)
